# Módulo 1: Fundamentos de Probabilidad e Inferencia Bayesiana — Práctica

**Módulo**: `1` | **Tipo**: PRÁCTICO | **Fecha**: 2026-05-20

In [1]:
import sys
import time
import numpy as np
import scipy
from scipy import stats
import plotly.graph_objects as go

np.random.seed(42)
assert sys.version_info >= (3, 12), f"Python 3.12+ requerido, tienes {sys.version}"
_t0 = time.time()
print(f"Python {sys.version_info.major}.{sys.version_info.minor}  |  NumPy {np.__version__}  |  SciPy {scipy.__version__}  ✓")

Python 3.14  |  NumPy 2.4.6  |  SciPy 1.17.1  ✓


In [2]:
# Generación del experimento: lanzamientos de moneda sesgada
# theta_real es el sesgo verdadero — el estudiante no lo conoce, el modelo lo estima.
rng = np.random.default_rng(42)
THETA_REAL = 0.7    # sesgo real de la moneda (oculto para el estimador)
N_OBS = 30          # número de lanzamientos

observaciones = rng.binomial(1, THETA_REAL, size=N_OBS)   # 1=cara, 0=sol
print(f"Lanzamientos realizados: {N_OBS}")
print(f"Secuencia: {observaciones}")
print(f"Total caras: {observaciones.sum()}   Total soles: {(1-observaciones).sum()}")

Lanzamientos realizados: 30
Secuencia: [0 1 0 1 1 0 0 0 1 1 1 0 1 0 1 1 1 1 0 1 0 1 0 0 0 1 1 1 1 1]
Total caras: 18   Total soles: 12


In [3]:
# [CELDA DE VALIDACIÓN — NO MODIFICAR]
# Cálculo de referencia analítico usando scipy.stats.beta
# Prior: Beta(2, 2)  →  Posterior tras observar las N_OBS monedas: Beta(α_n, β_n)

ALPHA_0, BETA_0 = 2.0, 2.0

k_caras = observaciones.sum()        # número total de caras
k_soles = N_OBS - k_caras            # número total de soles

alpha_ref = ALPHA_0 + k_caras
beta_ref  = BETA_0  + k_soles

# Estadísticos de referencia
posterior_mean_ref = stats.beta.mean(alpha_ref, beta_ref)
posterior_std_ref  = stats.beta.std(alpha_ref, beta_ref)
posterior_mode_ref = (alpha_ref - 1) / (alpha_ref + beta_ref - 2)

print(f"=== REFERENCIA ANALÍTICA ===")
print(f"Prior: Beta({ALPHA_0:.0f}, {BETA_0:.0f})")
print(f"Caras: {k_caras}   Soles: {k_soles}")
print(f"Posterior: Beta({alpha_ref:.0f}, {beta_ref:.0f})")
print(f"Media posterior:   {posterior_mean_ref:.6f}")
print(f"Desv. estándar:    {posterior_std_ref:.6f}")
print(f"Moda posterior:    {posterior_mode_ref:.6f}")

=== REFERENCIA ANALÍTICA ===
Prior: Beta(2, 2)
Caras: 18   Soles: 12
Posterior: Beta(20, 14)
Media posterior:   0.588235
Desv. estándar:    0.083189
Moda posterior:    0.593750


## Implementación: Actualización Bayesiana con moneda sesgada

**Objetivo**: Estimar la probabilidad de cara $\theta$ de una moneda sesgada, partiendo de un prior $\theta \sim \mathrm{Beta}(\alpha_0, \beta_0)$ y actualizando secuencialmente con cada observación.

**Recordatorio del modelo**:
- Prior: $\theta \sim \mathrm{Beta}(\alpha_0, \beta_0)$
- Observación $x_t \in \{0,1\}$ actualiza: $\alpha \mathrel{+}= x_t$, $\beta \mathrel{+}= (1 - x_t)$
- Posterior: $\theta \mid x_{1:t} \sim \mathrm{Beta}(\alpha_0 + \sum x_i,\; \beta_0 + t - \sum x_i)$

Completa el código a continuación.

In [4]:
# TODO: implementar la actualización bayesiana secuencial
# Parte 1 — Inicializar hiperparámetros del prior
alpha = ALPHA_0   # hiperparámetro α inicial
beta  = BETA_0    # hiperparámetro β inicial

# Historial para graficar
historia_alpha = [alpha]
historia_beta  = [beta]

# Parte 2 — Actualizar secuencialmente con cada observación
for x in observaciones:
    # TODO: actualizar alpha y beta según la observación x ∈ {0, 1}
    # alpha += ...
    # beta  += ...
    historia_alpha.append(alpha)
    historia_beta.append(beta)

# Parte 3 — Calcular estadísticos del posterior final
# TODO: calcula posterior_mean_student = alpha / (alpha + beta)
posterior_mean_student = alpha / (alpha + beta)  # completar

print(f"Tu posterior: Beta({alpha:.0f}, {beta:.0f})")
print(f"Media posterior (tu implementación): {posterior_mean_student:.6f}")

Tu posterior: Beta(2, 2)
Media posterior (tu implementación): 0.500000


In [5]:
# Validación numérica — descomentar cuando hayas completado los TODO
# TOL = 1e-4
# assert abs(posterior_mean_student - posterior_mean_ref) < TOL, (
#     f"Media posterior {posterior_mean_student:.6f} difiere de referencia "
#     f"{posterior_mean_ref:.6f} (diferencia = {abs(posterior_mean_student - posterior_mean_ref):.2e})"
# )
# assert alpha == alpha_ref, f'α_student={alpha} ≠ α_ref={alpha_ref}'
# assert beta  == beta_ref,  f'β_student={beta} ≠ β_ref={beta_ref}'
print("→ Descomenta las aserciones cuando hayas implementado los TODO.")
print(f"  Referencia: Beta({alpha_ref:.0f}, {beta_ref:.0f}), media = {posterior_mean_ref:.6f}")
print(f"  Tu impl.:   Beta({alpha:.1f}, {beta:.1f}), media = {posterior_mean_student:.6f}")

→ Descomenta las aserciones cuando hayas implementado los TODO.
  Referencia: Beta(20, 14), media = 0.588235
  Tu impl.:   Beta(2.0, 2.0), media = 0.500000


In [6]:
# Visualización Plotly — evolución del posterior Beta(α, β) con cada observación
# Nota: usa los valores de referencia para mostrar la curva correcta
# (cuando completes los TODO, historia_alpha/beta vendrán de tu implementación)
theta_grid = np.linspace(0.01, 0.99, 500)

# Historia de referencia para visualización
hist_a, hist_b = [ALPHA_0], [BETA_0]
_a, _b = ALPHA_0, BETA_0
for _x in observaciones:
    _a += _x
    _b += (1 - _x)
    hist_a.append(_a)
    hist_b.append(_b)

fig = go.Figure()
puntos  = [0, 5, 15, N_OBS]
colores = ["#636EFA", "#EF553B", "#00CC96", "#AB63FA"]
etiquetas = [f"Prior β({ALPHA_0:.0f},{BETA_0:.0f})", "t=5", "t=15", f"Posterior (t={N_OBS})"]

for i, (t, color, label) in enumerate(zip(puntos, colores, etiquetas)):
    a = hist_a[t]
    b = hist_b[t]
    densidad = stats.beta.pdf(theta_grid, a, b)
    fig.add_trace(go.Scatter(
        x=theta_grid, y=densidad,
        mode="lines",
        name=label,
        line=dict(color=color, width=2 + (i == len(puntos)-1)),
    ))

fig.add_vline(x=THETA_REAL, line_dash="dash", line_color="black",
              annotation_text=f"θ real = {THETA_REAL}", annotation_position="top right")

fig.update_layout(
    title="Módulo 1 — Evolución del posterior β(α, β) con observaciones",
    xaxis_title="θ (probabilidad de cara)",
    yaxis_title="P(θ | datos)",
    legend_title="Paso t",
    template="plotly_white",
    width=750, height=450,
)
fig.show()

## Ejercicios guiados (base, ≤ 30 min)

**Ejercicio 1**: Cambia el prior a $\mathrm{Beta}(10, 10)$ (prior más informativo centrado en 0.5) y repite la actualización. ¿Cuántas observaciones se necesitan para que la media posterior sea mayor que 0.65?

**Pista**: modifica `ALPHA_0` y `BETA_0` al principio de la celda anterior y vuelve a ejecutar.

**Ejercicio 2**: ¿Qué pasa si el prior es $\mathrm{Beta}(1, 1)$ (uniforme)? ¿En qué converge el posterior después de 30 observaciones?

**Ejercicio 3**: Verifica numéricamente que el posterior con $n=0$ observaciones es igual al prior. Usa los valores de `ALPHA_0`, `BETA_0` originales.

In [7]:
# Ejercicio 3 — verificación de identidad (prior = posterior con n=0)
alpha_test = ALPHA_0 + 0   # n=0 observaciones
beta_test  = BETA_0  + 0

assert alpha_test == ALPHA_0
assert beta_test  == BETA_0
print(f"Con n=0 observaciones: Beta({alpha_test:.0f}, {beta_test:.0f}) == Prior Beta({ALPHA_0:.0f}, {BETA_0:.0f})  ✓")
print("El prior es el caso especial del posterior sin datos.")

Con n=0 observaciones: Beta(2, 2) == Prior Beta(2, 2)  ✓
El prior es el caso especial del posterior sin datos.


## Reto (opcional — sin andamiaje)

Implementa la estimación bayesiana de la **media de una Gaussiana** con prior conjugado:

- Prior: $\mu \sim \mathcal{N}(0, 2^2)$
- Datos: $x_i \sim \mathcal{N}(\mu_{\text{real}}, 0.5^2)$ con $\mu_{\text{real}} = 1.2$, $n=25$ observaciones

(a) Calcula el posterior analítico $\mu_n$ y $\sigma_n^2$ usando las fórmulas de la sección de teoría.

(b) Genera 100 muestras del posterior usando `scipy.stats.norm.rvs` y verifica que su media empírica difiere en menos de 0.05 de $\mu_n$.

(c) Grafica con Plotly: prior, likelihood escalada, y posterior sobre $\mu \in [-1, 3]$.

In [8]:
# Smoke-test: verificar tiempo de ejecución total
_tiempo = time.time() - _t0
assert _tiempo < 30.0, f"Notebook tardó {_tiempo:.1f}s (máx 30s)"
print(f"Tiempo total de ejecución: {_tiempo:.2f}s  ✓")

Tiempo total de ejecución: 4.24s  ✓


## ¿Qué aprendiste?

Programando este notebook aprendimos que:

1. El ciclo *prior → likelihood → posterior* se implementa con **operaciones aritméticas simples** en el modelo conjugado Beta-Binomial: $\alpha \mathrel{+}= x$, $\beta \mathrel{+}= (1-x)$.
2. Con pocas observaciones el posterior es amplio (alta incertidumbre); con más datos se estrecha y converge hacia el valor real de $\theta$.
3. El prior informativo "resiste" más al cambio — necesita más datos para ser dominado por el likelihood.

## ¿Qué sigue?

En el **Módulo 2** extenderemos este ciclo a un espacio de estados discreto: en lugar de actualizar una distribución Beta(α, β) sobre un escalar $\theta$, mantendremos una distribución de probabilidad sobre un vector de $N$ celdas espaciales, y la actualizaremos con predicción (*Chapman-Kolmogorov*) y corrección (*update*) en cada paso de tiempo.